# Day 1 — Storage Layer Optimization
### OPTIMIZE · Z-ORDER · Liquid Clustering (incl. AUTO) · VACUUM · Broadcast Joins · Deletion Vectors & Predictive I/O · Clones

**Domain:** QuickBite QSR — a multi-store food & beverage chain. We build one
`orders` table deliberately as a pile of tiny files, and one small `stores`
dimension table. Every technique fixes something real and visible in this dataset.

**Runs on Databricks Free Edition serverless compute.** That has real consequences
for how a few sections work, confirmed by actually running this notebook:
- Managed Unity Catalog tables don't expose a physical file path — `dbutils.fs.ls`
  and reading raw `_delta_log/*.json` files **don't work**. Every inspection in this
  notebook goes through `DESCRIBE DETAIL` / `DESCRIBE HISTORY` instead — which,
  usefully, also works identically on classic compute, so nothing here is a
  downgrade if you do have a classic cluster.
- `VACUUM` enforces a **hard 168-hour (7-day) minimum retention with no override** —
  stronger than classic compute's configurable safety check. You cannot force a
  visible deletion in this environment. Section E's VS Code companion is the only
  place in this course where you actually watch VACUUM delete a file.
- **Free Edition has no Spark UI at all** (serverless-only compute). Every section
  below has a **VS Code Spark UI companion** — small, fast, self-contained code you
  copy into a local session to see the real Jobs/Stages/SQL tabs for that exact
  technique, with what to expect **before and after**.

**Reference:** Databricks, *"Comprehensive Guide to Optimize Databricks, Spark and
Delta Lake Workloads"*, plus current Delta Lake / Unity Catalog documentation.

## Setup

In [0]:
dbutils.widgets.text("catalog", "main", "Unity Catalog catalog")
dbutils.widgets.text("schema", "optimization_demo", "Schema (will be created)")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast
from functools import reduce
import time

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

ORDERS = f"{catalog}.{schema}.orders"
STORES = f"{catalog}.{schema}.stores"
ORDERS_LC = f"{catalog}.{schema}.orders_lc"
ORDERS_LC_AUTO = f"{catalog}.{schema}.orders_lc_auto"
ORDERS_CLONE = f"{catalog}.{schema}.orders_shallow_clone"

print(f"catalog.schema = {catalog}.{schema}")

catalog.schema = main.optimization_demo


### Two small helpers we reuse all day

In [0]:
snapshots = []
timings = []


def snapshot(table_name: str, stage_label: str):
    """Capture numFiles / size for table_name via DESCRIBE DETAIL and label it."""
    row = (
        spark.sql(f"DESCRIBE DETAIL {table_name}")
        .select("numFiles", "sizeInBytes")
        .withColumn("avg_file_size_mb", F.round(F.col("sizeInBytes") / F.col("numFiles") / 1024 / 1024, 3))
        .withColumn("stage", F.lit(stage_label))
        .select("stage", "numFiles", "sizeInBytes", "avg_file_size_mb")
    )
    snapshots.append(row)
    return row


def show_snapshots():
    display(reduce(lambda a, b: a.unionByName(b), snapshots))


def timed(label: str, fn):
    t0 = time.time()
    result = fn()
    elapsed = time.time() - t0
    timings.append((label, elapsed))
    print(f"{label}: {elapsed:.2f}s")
    return result


def show_timings():
    display(spark.createDataFrame(timings, ["stage", "elapsed_seconds"]))


def safe_conf(key: str, fallback: str) -> str:
    """spark.conf.get() isn't accessible for every key on serverless — fall back cleanly."""
    try:
        return spark.conf.get(key)
    except Exception:
        return fallback


## 🖥️ VS Code Spark UI companion — one-time setup

Every section below has its own companion cell, but they all share **one local
session** you set up once. Full install steps: `VS_Code_Spark_UI_Setup.md`.

**In a fresh venv (do NOT reuse a venv that already has `pyspark==4.2.0` — see the
note below):**
```bash
python3 -m venv venv && source venv/bin/activate
pip install pyspark==3.5.3 delta-spark==3.2.1 pandas ipython
```
This exact pair is confirmed to install and import cleanly together. The newest
`pyspark`/`delta-spark` releases have an active, unresolved version-compatibility
bug between them — this older pinned pair sidesteps it entirely, and comfortably
supports every command used today (`ZORDER` since Delta 2.0, `CLUSTER BY` since
Delta 3.1, `SHALLOW CLONE` is OSS-supported; `DEEP CLONE`, Deletion Vectors, and
Predictive I/O are Databricks-managed only — flagged where they come up).

**Paste this once into a new `.py` file in VS Code** (`# %%` cells — see the setup
guide for how to run them one at a time):

```python
# %%
import os, shutil, time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast
from delta import configure_spark_with_delta_pip

BASE_DIR = os.path.expanduser("~/spark-ui-test/day1_companion")
os.makedirs(BASE_DIR, exist_ok=True)
# Uncomment for a completely fresh start:
# shutil.rmtree(BASE_DIR, ignore_errors=True); os.makedirs(BASE_DIR, exist_ok=True)

builder = (
    SparkSession.builder.appName("Day1SparkUICompanion")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

ORDERS_PATH = f"{BASE_DIR}/orders"
STORES_PATH = f"{BASE_DIR}/stores"
ORDERS = f"delta.`{ORDERS_PATH}`"
STORES = f"delta.`{STORES_PATH}`"

# A SMALL dataset on purpose — this companion is about seeing the Spark UI signal
# for each technique quickly, not replicating Databricks' full-scale numbers.
store_rows = [(sid, f"Store {sid}", ["South","West","North","East"][sid % 4]) for sid in list(range(1, 60)) + [101]]
spark.createDataFrame(store_rows, ["store_id", "store_name", "region"]).write.format("delta").mode("overwrite").save(STORES_PATH)

def make_batch(batch_id, n=2000):
    return (
        spark.range(n)
        .withColumn("order_id", F.concat(F.lit(f"B{batch_id:03d}-"), F.col("id").cast("string")))
        .withColumn("store_id", F.when(F.rand() < 0.40, F.lit(101)).otherwise((F.rand()*59+1).cast("int")))
        .withColumn("customer_id", (F.rand()*50000).cast("int"))
        .withColumn("order_amount", F.round(F.rand()*45+5, 2))
        .drop("id")
    )

t0 = time.time()
for b in range(25):   # 25 small batches ~ 50,000 rows, seconds not minutes
    make_batch(b).write.format("delta").mode("append").save(ORDERS_PATH)
print(f"Built local orders table: {time.time()-t0:.1f}s, {spark.read.format('delta').load(ORDERS_PATH).count():,} rows")
```

Keep this session running (same Interactive Window / kernel) as you work through the
companion cells below — each one assumes `spark`, `ORDERS`, `ORDERS_PATH`, `STORES`,
`STORES_PATH`, and `make_batch` already exist.

## Section A — The Small Files Problem

Underneath every Delta table are plain Parquet files plus a `_delta_log` of
JSON/checkpoint transaction logs. Query performance is very sensitive to Parquet
file size: too many tiny files means Spark spends more time *opening and closing*
files than reading data. Healthy range: **16MB–1GB per file**.

We simulate the classic cause: frequent small appends, no file-size tuning. We also
bake in a deliberate skew — `store_id = 101` ("Flagship — MG Road") gets ~40% of all
orders — used later for the skew lab in Day 2.

### A.1 — Build the `stores` dimension table (small, 60 stores)

In [0]:
CITIES = ["Bengaluru", "Mumbai", "Delhi", "Pune", "Chennai", "Hyderabad", "Kolkata", "Ahmedabad"]
REGIONS = ["South", "West", "North", "East"]
STORE_TYPES = ["Dine-in", "Drive-thru", "Kiosk", "Delivery-only"]

store_rows = [
    (sid, f"Store {sid}" if sid != 101 else "Flagship - MG Road",
     CITIES[sid % len(CITIES)], REGIONS[sid % len(REGIONS)], STORE_TYPES[sid % len(STORE_TYPES)])
    for sid in list(range(1, 60)) + [101]
]

(
    spark.createDataFrame(store_rows, ["store_id", "store_name", "city", "region", "store_type"])
    .write.format("delta").mode("overwrite").saveAsTable(STORES)
)
display(spark.table(STORES).limit(5))

store_id,store_name,city,region,store_type
1,Store 1,Mumbai,West,Drive-thru
2,Store 2,Delhi,North,Kiosk
3,Store 3,Pune,East,Delivery-only
4,Store 4,Chennai,South,Dine-in
5,Store 5,Hyderabad,West,Drive-thru


### A.2 — Build the `orders` fact table as many small, unoptimized appends

200 append batches x ~5,000 rows = ~1,000,000 rows, each its own commit. **Takes a
few minutes.**

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {ORDERS} (
  order_id STRING, store_id INT, customer_id INT, order_date DATE,
  order_amount DOUBLE, item_count INT, product_category STRING, payment_type STRING
)
USING DELTA
""")

CATEGORIES = ["Burgers", "Fried Chicken", "Beverages", "Desserts", "Sides", "Breakfast"]
PAYMENT_TYPES = ["Card", "UPI", "Cash", "Wallet"]
NUM_BATCHES = 200
ROWS_PER_BATCH = 5000
FLAGSHIP_STORE_ID = 101
FLAGSHIP_WEIGHT = 0.40
NUM_CUSTOMERS = 500000


def make_batch(batch_id: int):
    return (
        spark.range(ROWS_PER_BATCH)
        .withColumn("order_id", F.concat(F.lit(f"B{batch_id:04d}-"), F.col("id").cast("string")))
        .withColumn("store_id", F.when(F.rand() < FLAGSHIP_WEIGHT, F.lit(FLAGSHIP_STORE_ID)).otherwise((F.rand()*59+1).cast("int")))
        .withColumn("customer_id", (F.rand() * NUM_CUSTOMERS).cast("int"))
        .withColumn("order_date", F.date_sub(F.current_date(), (F.rand() * 90).cast("int")))
        .withColumn("order_amount", F.round(F.rand() * 45 + 5, 2))
        .withColumn("item_count", (F.rand() * 5 + 1).cast("int"))
        .withColumn("product_category", F.element_at(F.array(*[F.lit(c) for c in CATEGORIES]), (F.rand()*len(CATEGORIES)).cast("int")+1))
        .withColumn("payment_type", F.element_at(F.array(*[F.lit(p) for p in PAYMENT_TYPES]), (F.rand()*len(PAYMENT_TYPES)).cast("int")+1))
        .drop("id")
    )


start = time.time()
for batch in range(NUM_BATCHES):
    make_batch(batch).write.format("delta").mode("append").saveAsTable(ORDERS)
    if batch % 25 == 0:
        print(f"  batch {batch:>3}/{NUM_BATCHES}  ({time.time() - start:6.1f}s elapsed)")

print(f"Done in {time.time() - start:.1f}s. Row count: {spark.table(ORDERS).count():,}")
snapshot(ORDERS, "A - before any optimization")

  batch   0/200  (   7.8s elapsed)
  batch  25/200  (  89.8s elapsed)
  batch  50/200  ( 169.4s elapsed)
  batch  75/200  ( 246.6s elapsed)
  batch 100/200  ( 319.6s elapsed)
  batch 125/200  ( 390.9s elapsed)
  batch 150/200  ( 457.9s elapsed)
  batch 175/200  ( 525.1s elapsed)
Done in 591.2s. Row count: 4,000,000


DataFrame[stage: string, numFiles: bigint, sizeInBytes: bigint, avg_file_size_mb: double]

### A.3 — Inspect the damage

For managed Unity Catalog tables, the physical file location isn't exposed —
`DESCRIBE DETAIL` is not a fallback here, it's the **only** tool, and it's all you
need: `numFiles`, `sizeInBytes`, and the derived average tell the whole story.

In [0]:
detail = spark.sql(f"DESCRIBE DETAIL {ORDERS}").select("numFiles", "sizeInBytes").collect()[0]
avg_file_size_kb = detail.sizeInBytes / detail.numFiles / 1024 if detail.numFiles else 0

print(f"{detail.numFiles} physical parquet files, {detail.sizeInBytes/1024/1024:.1f} MB total")
print(f"Average file size: {avg_file_size_kb:.1f} KB")
print(f"Target range: 16,384 KB (16MB) to 1,048,576 KB (1GB) per file")
print(f"These files are {'BELOW' if avg_file_size_kb < 16384 else 'within'} the healthy range.")

display(spark.sql(f"DESCRIBE DETAIL {ORDERS}"))

1745 physical parquet files, 34.5 MB total
Average file size: 20.3 KB
Target range: 16,384 KB (16MB) to 1,048,576 KB (1GB) per file
These files are BELOW the healthy range.


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,2b7d7f38-09c5-4da8-9517-cd0f08f0766f,main.optimization_demo.orders,null,,2026-08-22T07:02:39.424Z,2026-08-29T06:00:01.000Z,List(),List(),1745,36192706,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.autoOptimize.autoCompact -> false, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.autoOptimize.optimizeWrite -> false, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-f6a53e52-f866-4de6-9c0e-26bd5c48a21f, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-45898858-49f9-4d57-b0ad-77eefe830b70)",3,7,"List(appendOnly, deletionVectors, domainMetadata, invariants, rowTracking)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


### 🖥️ VS Code Spark UI companion — small files & the Environment tab

Free Edition has no Environment tab to show active Spark confs live — this is
exactly where a local session earns its keep.

```python
# %%
detail = spark.sql(f"DESCRIBE DETAIL {ORDERS}").select("numFiles", "sizeInBytes").collect()[0]
print(f"{detail.numFiles} files, {detail.sizeInBytes/1024:.0f} KB total")
```
**What to see in `localhost:4040`:** open the **Environment tab** — every active
Spark conf, live, with no code at all. This is the tab to reach for instead of
printing `spark.conf.get(...)` anywhere in this notebook.

### A.4 — Understanding the Delta Transaction Log (`_delta_log`)

Every technique today writes to the same structure: `_delta_log`, sitting next to
the Parquet files. One JSON file per table **version**; each is a sequence of
**actions** — `add` (a file was added, with min/max `stats`), `remove` (logically
dropped, physically kept until `VACUUM`), `commitInfo` (the operation + its metrics
— exactly what `DESCRIBE HISTORY` reads and formats). Nothing mutates in place —
every operation appends a new commit, which is what gives Delta ACID transactions and
time travel. Commits collapse into a checkpoint every 10 by default
(`delta.checkpointInterval`).

On managed UC tables we can't read these JSON files directly — `DESCRIBE HISTORY`
reads the exact same `commitInfo` action and formats it as a table.

In [0]:
history_df = spark.sql(f"DESCRIBE HISTORY {ORDERS}")
total_commits = history_df.count()
print(f"{total_commits} total commits (versions). Checkpoint every 10 by default -> ~{total_commits // 10} expected.")

print("\nMost recent commit (our last append batch):")
display(
    history_df.orderBy(F.desc("version")).limit(1)
    .select("version", "timestamp", "operation", "operationParameters", "operationMetrics")
)

808 total commits (versions). Checkpoint every 10 by default -> ~80 expected.

Most recent commit (our last append batch):


version,timestamp,operation,operationParameters,operationMetrics
807,2026-08-29T06:00:01.000Z,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])","Map(numFiles -> 8, numOutputRows -> 5000, numOutputBytes -> 69754)"


### A.5 — Baseline query performance (before optimization)

In [0]:
sample_customer_id = spark.table(ORDERS).select("customer_id").limit(1).collect()[0].customer_id
print(f"Sample customer_id for our running filter query: {sample_customer_id}")

baseline_count = timed(
    "A.5 baseline (before optimize/zorder)",
    lambda: spark.sql(f"SELECT * FROM {ORDERS} WHERE customer_id = {sample_customer_id}").count(),
)

Sample customer_id for our running filter query: 238730
A.5 baseline (before optimize/zorder): 1.54s


### 🖥️ VS Code Spark UI companion — baseline scan

```python
# %%
sample_customer_id = spark.read.format("delta").load(ORDERS_PATH).select("customer_id").limit(1).collect()[0].customer_id
spark.sql(f"SELECT * FROM {ORDERS} WHERE customer_id = {sample_customer_id}").count()
```
**Before you run C's companion below**, open `localhost:4040` → **SQL/DataFrame
tab** → click this query → expand the Delta scan node → **write down "number of
files read"**. This is your BEFORE number — Z-ORDER's companion gives you the AFTER.

## Section B — OPTIMIZE (Compaction / Bin-Packing)

| Command | Default target file size |
|---|---|
| Manual `OPTIMIZE` | **1 GB** (`delta.targetFileSize`) |
| Auto Optimize (Databricks-managed) | **128 MB** |

**Auto Optimize** has two independent parts: **Optimize Write** reshapes partition
sizes *during* the write; **Auto Compact** runs as a *separate* job right after.

**Production guidance:** run `OPTIMIZE` on its own job, not inside the ingestion job.

In [0]:
optimize_result = spark.sql(f"OPTIMIZE {ORDERS}")
display(optimize_result.select("metrics.*"))

snapshot(ORDERS, "B - after OPTIMIZE")
show_snapshots()

numFilesAdded,numFilesRemoved,filesAdded,filesRemoved,partitionsOptimized,zOrderStats,clusteringStats,numBins,numBatches,totalConsideredFiles,totalFilesSkipped,preserveInsertionOrder,numFilesSkippedToReduceWriteAmplification,numBytesSkippedToReduceWriteAmplification,startTimeMs,endTimeMs,totalClusterParallelism,totalScheduledTasks,autoCompactParallelismStats,deletionVectorStats,recompressionCodec,numTableColumns,numTableColumnsWithStats,totalTaskExecutionTimeMs,skippedArchivedFiles,clusteringMetrics,fileAgeHistogram,numFilesSortedWithinFiles
1,1745,"List(28824271, 28824271, 2.8824271E7, 1, 28824271)","List(8561, 21003726, 20740.805730659027, 1745, 36192706)",0,null,null,0,1,1745,0,true,0,0,1787983285128,1787983293476,8,1,null,"List(0, 0)",null,8,8,2599,0,null,null,0


stage,numFiles,sizeInBytes,avg_file_size_mb
A - before any optimization,1745,36192706,0.02
B - after OPTIMIZE,1,28824271,27.489


### B.1 — Cross-check with `DESCRIBE HISTORY`

In [0]:
display(
    spark.sql(f"DESCRIBE HISTORY {ORDERS}")
    .where("operation = 'OPTIMIZE'").orderBy(F.desc("version")).limit(1)
    .select("version", "timestamp", "operationParameters.zOrderBy",
            "operationMetrics.numRemovedFiles", "operationMetrics.numAddedFiles",
            "operationMetrics.numRemovedBytes", "operationMetrics.numAddedBytes")
)

version,timestamp,zOrderBy,numRemovedFiles,numAddedFiles,numRemovedBytes,numAddedBytes
808,2026-08-29T06:01:34.000Z,[],1745,1,36192706,28824271


### 🖥️ VS Code Spark UI companion — OPTIMIZE

```python
# %%
before = spark.sql(f"DESCRIBE DETAIL {ORDERS}").select("numFiles").collect()[0].numFiles
spark.sql(f"OPTIMIZE {ORDERS}")
after = spark.sql(f"DESCRIBE DETAIL {ORDERS}").select("numFiles").collect()[0].numFiles
print(f"numFiles: {before} -> {after}")
```
**What to see — BEFORE:** `localhost:4040` → **Jobs tab** is either empty or shows
only the 25 small append-batch jobs from setup, each tiny.
**AFTER running this cell:** a new job appears. Click it → **Stages tab** → almost
entirely **file I/O time**, close to **zero shuffle read/write** — compaction
rewrites files, it never redistributes rows by key. Compare this stage's shape
against Section F's broadcast join later, which looks completely different.

## Section C — Z-ORDER

`ZORDER BY` physically co-locates rows with similar values, so Delta's min/max
data-skipping stats (first 32 columns, automatic) become far more selective.

**Column selection:** high-cardinality, filter/join columns — `customer_id` (500,000
distinct values) is textbook. **Never Z-ORDER on more than 4 columns.**

> Databricks recommends *not* partitioning tables under 1TB — why `orders` stays
> unpartitioned here. Day 2 revisits partitioning explicitly for the DPP/DFP lab.

In [0]:
zorder_result = spark.sql(f"OPTIMIZE {ORDERS} ZORDER BY (customer_id)")
display(zorder_result.select("metrics.*"))

snapshot(ORDERS, "C - after OPTIMIZE ZORDER BY")
show_snapshots()

numFilesAdded,numFilesRemoved,filesAdded,filesRemoved,partitionsOptimized,zOrderStats,clusteringStats,numBins,numBatches,totalConsideredFiles,totalFilesSkipped,preserveInsertionOrder,numFilesSkippedToReduceWriteAmplification,numBytesSkippedToReduceWriteAmplification,startTimeMs,endTimeMs,totalClusterParallelism,totalScheduledTasks,autoCompactParallelismStats,deletionVectorStats,recompressionCodec,numTableColumns,numTableColumnsWithStats,totalTaskExecutionTimeMs,skippedArchivedFiles,clusteringMetrics,fileAgeHistogram,numFilesSortedWithinFiles
0,0,"List(null, null, 0.0, 0, 0)","List(null, null, 0.0, 0, 0)",0,"List(minCubeSize(107374182400), List(0, 0), List(1, 28824271), 0, List(0, 0), 0, null)",null,0,0,1,1,false,0,0,1787983328902,1787983332376,8,0,null,"List(0, 0)",null,8,8,0,0,null,null,0


stage,numFiles,sizeInBytes,avg_file_size_mb
A - before any optimization,1745,36192706,0.02
B - after OPTIMIZE,1,28824271,27.489
C - after OPTIMIZE ZORDER BY,1,28824271,27.489


### C.1 — ZCube stats from `DESCRIBE HISTORY`

On managed UC tables we can't read per-file `ZCUBE_ID` tags from the raw log — but
`DESCRIBE HISTORY`'s `operationMetrics` gives the aggregate picture: how many files
were already in a ZCube (`totalConsideredFiles` vs. skipped) before this run touched
them at all — the mechanism behind **incremental Z-Ordering**.

In [0]:
zorder_history = (
    spark.sql(f"DESCRIBE HISTORY {ORDERS}")
    .where("operation = 'OPTIMIZE' AND operationParameters.zOrderBy is not null")
    .orderBy(F.desc("version")).limit(1)
)
display(
    zorder_history.select(
        "version", "timestamp", "operationParameters.zOrderBy",
        "operationMetrics.numAddedFiles", "operationMetrics.numRemovedFiles",
        "operationMetrics.totalConsideredFiles", "operationMetrics.totalFilesSkipped",
    )
)
print("Try it live: re-run OPTIMIZE ... ZORDER BY (customer_id) with no new data written —")
print("totalConsideredFiles should drop near zero, since files are already ZCube-tagged.")

version,timestamp,zOrderBy,numAddedFiles,numRemovedFiles,totalConsideredFiles,totalFilesSkipped
808,2026-08-29T06:01:34.000Z,[],1,1745,null,null


Try it live: re-run OPTIMIZE ... ZORDER BY (customer_id) with no new data written —
totalConsideredFiles should drop near zero, since files are already ZCube-tagged.


### C.2 — Data skipping: does the same filter query read fewer files now?

In [0]:
after_zorder_count = timed(
    "C.2 after optimize + zorder",
    lambda: spark.sql(f"SELECT * FROM {ORDERS} WHERE customer_id = {sample_customer_id}").count(),
)
assert after_zorder_count == baseline_count, "row count changed — investigate before trusting the timing comparison"
show_timings()

spark.sql(f"SELECT * FROM {ORDERS} WHERE customer_id = {sample_customer_id}").explain(mode="formatted")

C.2 after optimize + zorder: 1.00s


stage,elapsed_seconds
A.5 baseline (before optimize/zorder),1.5374293327331543
C.2 after optimize + zorder,1.0038251876831055


== Physical Plan ==
PhotonResultStage (3)
+- PhotonColumnarToRow (2)
   +- PhotonScan parquet main.optimization_demo.orders (1)


(1) PhotonScan parquet main.optimization_demo.orders
Output [8]: [order_id#11650, store_id#11651, customer_id#11652, order_date#11653, order_amount#11654, item_count#11655, product_category#11656, payment_type#11657]
DictionaryFilters: [(customer_id#11652 = 238730)]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-c8dht/uc/3369cc0e-8d2a-4615-9cf8-6eee7e01f690/f7b33d7b-bce6-4757-8ba5-3b931ab871f9/__unitystorage/catalogs/c11d551e-bcfa-4868-afe1-279c6cd98346/tables/97a45b5a-eab0-47cb-8941-d3cc085722b8]
ReadSchema: struct<order_id:string,store_id:int,customer_id:int,order_date:date,order_amount:double,item_count:int,product_category:string,payment_type:string>
RequiredDataFilters: [isnotnull(customer_id#11652), (customer_id#11652 = 238730)]

(2) PhotonColumnarToRow
Input [8]: [order_id#11650, store_id#11651, customer_id#11652, order_date#11653, order_amount

### 🖥️ VS Code Spark UI companion — Z-ORDER, before/after data skipping

```python
# %%
spark.sql(f"OPTIMIZE {ORDERS} ZORDER BY (customer_id)")
spark.sql(f"SELECT * FROM {ORDERS} WHERE customer_id = {sample_customer_id}").count()
```
**What to see:** SQL/DataFrame tab → this query's scan node → "number of files read"
— compare directly against the number you wrote down in A.5's companion. On this
small local dataset expect a visible drop, though the effect is far more dramatic at
the full 1,000,000-row scale you just ran in Databricks above. Also check
`PushedFilters` on the same node — same signal `.explain()` prints as text.

### C.3 — Cross-check with `DESCRIBE HISTORY`

In [0]:
display(
    spark.sql(f"DESCRIBE HISTORY {ORDERS}")
    .where("operation = 'OPTIMIZE'").orderBy(F.desc("version")).limit(1)
    .select("version", "timestamp", "operationParameters.zOrderBy",
            "operationMetrics.numRemovedFiles", "operationMetrics.numAddedFiles")
)

version,timestamp,zOrderBy,numRemovedFiles,numAddedFiles
808,2026-08-29T06:01:34.000Z,[],1745,1


## Section D — Liquid Clustering

`CLUSTER BY` uses the same ZCube mechanism as Z-Order, as a first-class, stateful
table property instead of a command you re-run:

| | Z-ORDER | Liquid Clustering |
|---|---|---|
| Invocation | `OPTIMIZE t ZORDER BY (cols)` every time | `CLUSTER BY (cols)` set once |
| Changing columns | Full table rewrite | `ALTER TABLE t CLUSTER BY (new_cols)` — only *new* data clusters on the new key |
| Max columns | 4 (hard guidance) | More flexible |
| Best fit | Existing tables, one-off reorganization | New tables, or evolving query patterns |

**When to use which** — Z-ORDER for existing tables you're reorganizing once, with a
stable set of ≤4 filter columns; Liquid Clustering for new tables, or whenever the
clustering key might change later. Databricks' own default recommendation for new UC
managed tables going forward is Liquid Clustering.

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {ORDERS_LC} (
  order_id STRING, store_id INT, customer_id INT, order_date DATE,
  order_amount DOUBLE, item_count INT, product_category STRING, payment_type STRING
)
USING DELTA
CLUSTER BY (customer_id, store_id)
""")

for lc_batch in range(3):
    make_batch(lc_batch).withColumn("order_id", F.concat(F.lit(f"LC{lc_batch}-"), F.col("order_id"))).write.format("delta").mode("append").saveAsTable(ORDERS_LC)

spark.sql(f"OPTIMIZE {ORDERS_LC}")
display(spark.sql(f"DESCRIBE DETAIL {ORDERS_LC}").select("clusteringColumns", "numFiles", "sizeInBytes"))
print(f"To re-cluster later without a rewrite: ALTER TABLE {ORDERS_LC} CLUSTER BY (store_id); OPTIMIZE {ORDERS_LC};")

clusteringColumns,numFiles,sizeInBytes
"List(customer_id, store_id)",1,315785


To re-cluster later without a rewrite: ALTER TABLE main.optimization_demo.orders_lc CLUSTER BY (store_id); OPTIMIZE main.optimization_demo.orders_lc;


### D.2 — `CLUSTER BY AUTO`: letting Databricks pick the clustering columns

Manual `CLUSTER BY (cols)` still requires you to know the right columns up front.
**`CLUSTER BY AUTO`** hands that decision to Databricks: it observes actual query
patterns (filters, joins) over time and adjusts clustering keys automatically,
without a full rewrite — the same "set once" simplicity as manual Liquid Clustering,
but without having to correctly guess the key in advance.

**When to use `AUTO` vs. an explicit key:** `AUTO` when query patterns are still
settling or vary across teams/dashboards; an explicit key when you already know
exactly which 1–4 columns dominate your filters and want deterministic, predictable
clustering behavior.

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {ORDERS_LC_AUTO} (
  order_id STRING, store_id INT, customer_id INT, order_date DATE,
  order_amount DOUBLE, item_count INT, product_category STRING, payment_type STRING
)
USING DELTA
CLUSTER BY AUTO
""")

for lc_batch in range(3):
    make_batch(lc_batch).withColumn("order_id", F.concat(F.lit(f"LCA{lc_batch}-"), F.col("order_id"))).write.format("delta").mode("append").saveAsTable(ORDERS_LC_AUTO)

# Run a few representative queries so Databricks has query patterns to learn from —
# AUTO doesn't pick a key on day one, it adapts as usage accumulates.
spark.sql(f"SELECT * FROM {ORDERS_LC_AUTO} WHERE customer_id = {sample_customer_id}").count()
spark.sql(f"SELECT * FROM {ORDERS_LC_AUTO} WHERE store_id = 101").count()

display(spark.sql(f"DESCRIBE DETAIL {ORDERS_LC_AUTO}").select("clusteringColumns", "numFiles"))
print("clusteringColumns may show empty until Databricks has observed enough query activity")
print("to choose a key — this is expected on a freshly created table with only a couple of queries run.")

clusteringColumns,numFiles
List(),24


clusteringColumns may show empty until Databricks has observed enough query activity
to choose a key — this is expected on a freshly created table with only a couple of queries run.


### 🖥️ VS Code Spark UI companion — Liquid Clustering

`CLUSTER BY AUTO` is a Databricks-managed decision (it needs observed workload
history) — no local equivalent. Manual `CLUSTER BY (cols)` **is** genuine OSS Delta
(3.1.0+), so that part of this section runs for real locally:

```python
# %%
ORDERS_LC_PATH = f"{BASE_DIR}/orders_lc"
ORDERS_LC = f"delta.`{ORDERS_LC_PATH}`"
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {ORDERS_LC} (
  order_id STRING, store_id INT, customer_id INT, order_amount DOUBLE
) USING DELTA CLUSTER BY (customer_id, store_id)
""")
for b in range(5):
    make_batch(b, n=2000).withColumn("order_id", F.concat(F.lit(f"LC{b}-"), F.col("order_id"))) \
        .write.format("delta").mode("append").save(ORDERS_LC_PATH)
spark.sql(f"OPTIMIZE {ORDERS_LC}")
```
**What to see:** same shape as Section B's companion — Jobs/Stages tab, mostly file
I/O, no shuffle. Liquid Clustering and plain OPTIMIZE look identical in the Spark
UI; the difference is entirely in what gets persisted as table metadata
(`clusteringColumns`), not in how the job executes.

## Section E — VACUUM

| Setting | Default |
|---|---|
| Data file retention | **7 days** (`delta.deletedFileRetentionDuration`) |
| Transaction log retention | **30 days** (`delta.logRetentionDuration`) |

**On this serverless environment specifically:** VACUUM enforces a **hard 168-hour
minimum with no override** — stronger than classic compute, where the check can be
(carefully) disabled. That means you genuinely cannot force a visible deletion here.
That's not a gap in this notebook — it's the platform correctly refusing to let a
training session do something unsafe. The VS Code companion below is where you
actually watch a file get deleted.

In [0]:
dry_run_files = spark.sql(f"VACUUM {ORDERS} DRY RUN").collect()
print(f"Files eligible for deletion at the default 7-day retention: {len(dry_run_files)}")
print("Expected: close to zero — the files OPTIMIZE/ZORDER just replaced are only minutes old.")

logical_before = spark.sql(f"DESCRIBE DETAIL {ORDERS}").collect()[0].numFiles
vacuum_result = spark.sql(f"VACUUM {ORDERS}")
display(vacuum_result)
logical_after = spark.sql(f"DESCRIBE DETAIL {ORDERS}").collect()[0].numFiles

print(f"\nLogical files (current version): {logical_before} -> {logical_after}  <- unchanged, as expected")
print("VACUUM only ever removes files the CURRENT version no longer references — never active ones.")
print("Note: on this platform, retention cannot be lowered below 168 hours (7 days), so nothing")
print("eligible for deletion exists yet — this run itself is a no-op, by design.")

### E.1 — Cross-check with `DESCRIBE HISTORY`

`VACUUM` shows up as **two** rows — `VACUUM START` (what it found eligible) and
`VACUUM END` (what it actually deleted).

In [0]:
display(
    spark.sql(f"DESCRIBE HISTORY {ORDERS}")
    .where("operation IN ('VACUUM START', 'VACUUM END')")
    .orderBy(F.desc("version")).limit(2)
    .select("version", "timestamp", "operation",
            "operationMetrics.numFilesToDelete", "operationMetrics.sizeOfDataToDelete",
            "operationMetrics.numDeletedFiles", "operationMetrics.numVacuumedDirectories")
)

### 🖥️ VS Code Spark UI companion — VACUUM, actually deleting a file

This is disposable local test data, so we can safely override the retention check —
something Free Edition deliberately won't let you do.

```python
# %%
import os

def physical_file_count(path):
    return len([f for f in os.scandir(path.replace("file:", "")) if f.name.endswith(".parquet")])

before_physical = physical_file_count(ORDERS_PATH)
spark.sql(f"OPTIMIZE {ORDERS}")   # creates files that immediately become superseded

spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
display_result = spark.sql(f"VACUUM {ORDERS} RETAIN 0 HOURS").collect()
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "true")   # always restore this

after_physical = physical_file_count(ORDERS_PATH)
print(f"Physical files: {before_physical} -> {after_physical}  ({before_physical - after_physical} removed)")
```
**What to see — BEFORE:** Jobs tab shows the OPTIMIZE job. **AFTER:** a new VACUUM
job appears — open it, almost entirely file-deletion I/O, no shuffle, similar shape
to OPTIMIZE's job but doing the opposite (removing files instead of writing them).
The printed before/after physical file count is the only place in this entire course
you see VACUUM's deletion happen in real time.

**Flag to learners: `RETAIN 0 HOURS` with the check disabled is training-only, on
disposable local data. Never do this against a production table.**

## Section F — Broadcast Joins

| Setting | Default |
|---|---|
| Static auto-broadcast threshold | **10 MB** (`spark.sql.autoBroadcastJoinThreshold`) |
| AQE runtime conversion threshold | **30 MB** |
| Hard limit | never > 1GB on disk; **8GB in-memory** cap |

**When to broadcast:** the small side is confidently under ~30MB, or you're
comfortable forcing it below 10MB with an explicit hint. Never broadcast over 1GB on
disk; stay well clear of the 8GB in-memory cap, especially on heavily-compressed data.

`spark.conf.get('spark.sql.autoBroadcastJoinThreshold')` isn't directly accessible on
serverless — we use `safe_conf()` from Setup instead of letting that crash the cell.

In [0]:
threshold_display = safe_conf('spark.sql.autoBroadcastJoinThreshold', '10485760 (10MB, serverless default)')
stores_size_kb = spark.sql(f"DESCRIBE DETAIL {STORES}").collect()[0].sizeInBytes / 1024
print(f"autoBroadcastJoinThreshold = {threshold_display}   |   stores table size = {stores_size_kb:.1f} KB")

autoBroadcastJoinThreshold = 10485760 (10MB, serverless default)   |   stores table size = 2.2 KB


In [0]:
# F.1 — Unhinted join: let Spark's planner decide
unhinted_join = spark.sql(f"""
    SELECT o.order_id, o.order_amount, s.store_name, s.region
    FROM {ORDERS} o JOIN {STORES} s ON o.store_id = s.store_id
""")
unhinted_join.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (10)
+- == Initial Plan ==
   PhotonResultStage (9)
   +- PhotonColumnarToRow (8)
      +- PhotonProject (7)
         +- PhotonBroadcastHashJoin Inner (6)
            :- PhotonScan parquet main.optimization_demo.orders (1)
            +- PhotonShuffleExchangeSource (5)
               +- PhotonShuffleMapStage (4)
                  +- PhotonShuffleExchangeSink (3)
                     +- PhotonScan parquet main.optimization_demo.stores (2)


(1) PhotonScan parquet main.optimization_demo.orders
Output [3]: [order_id#110944, store_id#110945, order_amount#110948]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-c8dht/uc/3369cc0e-8d2a-4615-9cf8-6eee7e01f690/f7b33d7b-bce6-4757-8ba5-3b931ab871f9/__unitystorage/catalogs/c11d551e-bcfa-4868-afe1-279c6cd98346/tables/97a45b5a-eab0-47cb-8941-d3cc085722b8]
ReadSchema: struct<order_id:string,store_id:int,order_amount:double>
RequiredDataFilters: [isnotnull(store_id#110945)]

(2) PhotonScan parquet main.optimi

In [0]:
# F.2 — Explicitly broadcast-hinted join
hinted_join = (
    spark.table(ORDERS).alias("o")
    .join(broadcast(spark.table(STORES)).alias("s"), on="store_id")
    .select("order_id", "order_amount", "store_name", "region")
)
hinted_join.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (10)
+- == Initial Plan ==
   PhotonResultStage (9)
   +- PhotonColumnarToRow (8)
      +- PhotonProject (7)
         +- PhotonBroadcastHashJoin Inner (6)
            :- PhotonScan parquet main.optimization_demo.orders (1)
            +- PhotonShuffleExchangeSource (5)
               +- PhotonShuffleMapStage (4)
                  +- PhotonShuffleExchangeSink (3)
                     +- PhotonScan parquet main.optimization_demo.stores (2)


(1) PhotonScan parquet main.optimization_demo.orders
Output [3]: [order_id#111139, store_id#111140, order_amount#111143]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-c8dht/uc/3369cc0e-8d2a-4615-9cf8-6eee7e01f690/f7b33d7b-bce6-4757-8ba5-3b931ab871f9/__unitystorage/catalogs/c11d551e-bcfa-4868-afe1-279c6cd98346/tables/97a45b5a-eab0-47cb-8941-d3cc085722b8]
ReadSchema: struct<order_id:string,store_id:int,order_amount:double>
RequiredDataFilters: [isnotnull(store_id#111140)]

(2) PhotonScan parquet main.optimi

### 🖥️ VS Code Spark UI companion — broadcast join

```python
# %%
from pyspark.sql.functions import broadcast
unhinted = spark.read.format("delta").load(ORDERS_PATH).join(spark.read.format("delta").load(STORES_PATH), "store_id")
unhinted.collect()
hinted = spark.read.format("delta").load(ORDERS_PATH).join(broadcast(spark.read.format("delta").load(STORES_PATH)), "store_id")
hinted.collect()
```
**What to see:** SQL/DataFrame tab, compare both queries' DAGs. `unhinted` may
already show `BroadcastExchange` (Spark's planner or AQE deciding for you) —
`hinted` should show it deterministically. Either way, neither should show a plain
`Exchange` on the `stores` side — that shuffle is exactly what broadcasting avoids.

### F.3 — Guardrails to remember
- Broadcast hash join is **not supported for full outer joins**
- Right outer join: only the **left** side can be broadcast; other left joins: only the **right**
- **Never broadcast a table larger than 1GB on disk**
- Hard **8GB in-memory** cap regardless of on-disk size — compression can blow past this silently

## Section G — Deletion Vectors & Predictive I/O
*(Databricks-managed, Photon-exclusive — conceptual only, no local Spark UI equivalent)*

**Deletion Vectors:** instead of rewriting an entire Parquet file when one row is
deleted or updated, Delta marks the deleted rows in a small bitmap file alongside the
original — the original stays untouched. Rewrites are deferred to the next
`OPTIMIZE`, which cleans up files with a lot of accumulated deletion-vector "noise".

**Predictive I/O:** exclusive to the **Photon** engine. Two parts —
**accelerated reads** (an ML model picks the most efficient scan/filter access
pattern) and **accelerated updates** (uses Deletion Vectors to avoid full-file
rewrites on `DELETE`/`UPDATE`/`MERGE`). Requires Photon-enabled compute or a
serverless/pro SQL warehouse, DBR 11.3 LTS+.

**Why no VS Code companion here:** both features are tied to Databricks' own
managed runtime and Photon specifically — there's no equivalent to install locally.
Treat this as a whiteboard topic, same as Predictive Optimization on Day 2.

In [0]:
try:
    spark.sql(f"ALTER TABLE {ORDERS} SET TBLPROPERTIES ('delta.enableDeletionVectors' = 'true')")
    display(spark.sql(f"SHOW TBLPROPERTIES {ORDERS}").where("key = 'delta.enableDeletionVectors'"))
except Exception as e:
    print(f"If this errors on your workspace/edition, that's expected — note the error and move on: {e}")

key,value
delta.enableDeletionVectors,true


## Section H — Shallow & Deep Clones

| | Shallow Clone | Deep Clone |
|---|---|---|
| Copies | Metadata only — references source's existing data files | Metadata **and** all data files |
| Speed/cost | Fast, cheap | Slower, full storage cost |
| Risk | Breaks if the source is `VACUUM`ed and referenced files are removed | Fully independent, safe from source changes |
| Best for | Dev/test/experimentation without duplicating storage | Backup, migration, disaster recovery |
| OSS Delta support | Yes | **No — Databricks-managed only** |

**When to use which:** shallow for a quick, disposable dev/test copy where you don't
mind it breaking if someone VACUUMs the source; deep when you need a fully
independent copy that survives the source table's lifecycle — backups, migrations,
cross-region copies.

In [0]:
spark.sql(f"CREATE OR REPLACE TABLE {ORDERS_CLONE} SHALLOW CLONE {ORDERS}")
display(spark.sql(f"DESCRIBE DETAIL {ORDERS_CLONE}").select("numFiles", "sizeInBytes"))
print(f"{ORDERS_CLONE} created — check numFiles/sizeInBytes above against {ORDERS} from Section F:")
print("nearly instant, and sizeInBytes reflects REFERENCED data, not a physical copy.")

numFiles,sizeInBytes
1,28824271


main.optimization_demo.orders_shallow_clone created — check numFiles/sizeInBytes above against main.optimization_demo.orders from Section F:
nearly instant, and sizeInBytes reflects REFERENCED data, not a physical copy.


### 🖥️ VS Code Spark UI companion — Shallow Clone

Genuinely OSS Delta-supported — Deep Clone is not, so there's no companion for that
half of this section.

```python
# %%
CLONE_PATH = f"{BASE_DIR}/orders_shallow_clone"
spark.sql(f"CREATE OR REPLACE TABLE delta.`{CLONE_PATH}` SHALLOW CLONE {ORDERS}")
```
**What to see:** Jobs tab — a very short-lived job, almost no I/O at all (no data is
copied, just metadata). Contrast this against Section B's OPTIMIZE companion job,
which does real file I/O — cloning is close to instantaneous specifically *because*
it skips that work.

## Section I — Edge Case: Watching a Query Fail in the Spark UI

Every section so far showed a technique working. Production systems also need you to
diagnose a technique or job that's **failing** — and this is the one class of
problem Free Edition genuinely cannot show you, since there's no Jobs/Stages view to
inspect a failure in. This section runs on Databricks so you see the error Spark
surfaces directly in the notebook — but the VS Code companion is where you actually
see what a **failed job looks like** in the Spark UI itself.

We use a Python UDF that divides by a column that's zero for some rows, unguarded —
deterministic, reliable, and a good callback to the general advice to avoid Python
UDFs where a native Spark function will do: a bug like this one is exactly the kind
of failure mode a Python UDF makes easy to introduce and hard to see coming.

In [0]:
from pyspark.sql.types import DoubleType


def risky_divide(order_amount, item_count):
    return order_amount / item_count   # no guard against item_count == 0 — the bug, on purpose


risky_divide_udf = F.udf(risky_divide, DoubleType())

try:
    result = (
        spark.table(ORDERS)
        .withColumn("item_count_broken", F.when(F.rand() < 0.001, F.lit(0)).otherwise(F.col("item_count")))
        .withColumn("amount_per_item", risky_divide_udf(F.col("order_amount"), F.col("item_count_broken")))
        .collect()
    )
    print("No failure this run — the 0.1% injected zero-rate is random. Re-run the cell if this happened.")
except Exception as e:
    print("Query failed, as expected. The exception below is what Spark surfaces on the driver side —")
    print("this notebook environment has no Spark UI to inspect the failed job/stage/task further:\n")
    print(str(e)[:1500])

Query failed, as expected. The exception below is what Spark surfaces on the driver side —
this notebook environment has no Spark UI to inspect the failed job/stage/task further:


  An exception was thrown from the Python worker. Please see the stack trace below.
[PYTHON_EXCEPTION] An exception was thrown from the Python worker: Traceback (most recent call last):
  File "/home/spark-2c90c619-6f9b-4937-961d-17/.ipykernel/75/command-8901078648257930-1722044491", line 5, in risky_divide
ZeroDivisionError: float division by zero
 SQLSTATE: 38000


### 🖥️ VS Code Spark UI companion — seeing the failure in Jobs/Stages

```python
# %%
from pyspark.sql.types import DoubleType

def risky_divide(order_amount, item_count):
    return order_amount / item_count   # unguarded — will raise ZeroDivisionError on item_count == 0

risky_divide_udf = F.udf(risky_divide, DoubleType())

broken = (
    spark.read.format("delta").load(ORDERS_PATH)
    .withColumn("item_count", F.when(F.rand() < 0.05, F.lit(0)).otherwise(F.lit(2)))   # 5% zeros — reliably triggers fast
    .withColumn("amount_per_item", risky_divide_udf(F.col("order_amount"), F.col("item_count")))
)
try:
    broken.collect()
except Exception as e:
    print("Failed as expected — now go look at the Spark UI.")
```
**What to see in `localhost:4040`:**
- **Jobs tab** — this job shows a **red/failed status**, unlike every successful job
  so far
- Click into it → **Stages tab** — the failed stage shows a **non-zero "Failed"
  count** in its task summary
- Click into the stage → the task table shows **individual failed tasks** — Spark
  retries a failed task automatically (`spark.task.maxFailures`, default **4**) before
  giving up on the stage, so expect to see **more task attempts than partitions**
- Click into one failed task → its detail panel includes the **full Python
  traceback** — the actual `ZeroDivisionError` and the line of your UDF that raised
  it, exactly the debugging information the driver-side exception in the Databricks
  cell above can't show you as clearly

This is the concrete case for keeping VS Code's Spark UI in your toolkit even once
you have classic Databricks compute: driver-side error messages tell you *that*
something failed; the Spark UI's failed-stage task table tells you *which partition,
which attempt, and why*.

### Full Operations Timeline

In [0]:
display(
    spark.sql(f"DESCRIBE HISTORY {ORDERS}")
    .where("operation IN ('OPTIMIZE', 'VACUUM START', 'VACUUM END')")
    .orderBy("version")
    .select("version", "timestamp", "operation", "operationParameters.zOrderBy",
            "operationMetrics.numRemovedFiles", "operationMetrics.numAddedFiles",
            "operationMetrics.numFilesToDelete", "operationMetrics.numDeletedFiles")
)

version,timestamp,operation,zOrderBy,numRemovedFiles,numAddedFiles,numFilesToDelete,numDeletedFiles
201,2026-08-22T07:23:06.000Z,OPTIMIZE,[],1600,1,null,null
402,2026-08-23T04:43:51.000Z,OPTIMIZE,[],1601,1,null,null
403,2026-08-23T05:04:41.000Z,VACUUM START,null,null,null,0,null
404,2026-08-23T05:04:42.000Z,VACUUM END,null,null,null,null,0
405,2026-08-23T05:05:42.000Z,VACUUM START,null,null,null,0,null
406,2026-08-23T05:05:42.001Z,VACUUM END,null,null,null,null,0
601,2026-08-29T04:42:28.000Z,OPTIMIZE,[],1457,1,null,null
808,2026-08-29T06:01:34.000Z,OPTIMIZE,[],1745,1,null,null


## Day 1 Recap

| Technique | What it fixes | Default(s) / when to use |
|---|---|---|
| `OPTIMIZE` | Small-file fragmentation | 1GB target (manual) / 128MB (Auto Optimize) |
| `OPTIMIZE ... ZORDER BY` | Poor data skipping | Max 4 cols; existing tables, stable filter columns |
| Liquid Clustering (`CLUSTER BY`) | Same as Z-Order, incremental | New tables, or evolving clustering keys |
| `CLUSTER BY AUTO` | Same, without choosing the key yourself | Query patterns still settling or vary by team |
| `VACUUM` | Storage bloat | 7-day retention (this env: 168h hard minimum, no override) |
| Broadcast joins | Unnecessary shuffle | 10MB static / 30MB AQE / 1GB-disk & 8GB-memory hard caps |
| Deletion Vectors + Predictive I/O | Full-file rewrites on DELETE/UPDATE/MERGE | Photon-only, DBR 11.3 LTS+ |
| Shallow Clone | Cheap dev/test copies | Fine with the copy breaking if source is VACUUMed |
| Deep Clone | Independent, durable copies | Backup/migration/DR — Databricks-managed only |

**On Databricks Free Edition specifically:** `DESCRIBE DETAIL`/`DESCRIBE HISTORY`
replace all direct file/log access; VACUUM can't be forced to delete visibly; several
`spark.conf.get()` calls need a fallback. None of this is a downgrade on classic
compute — it's a stricter, universally-portable way to write the same notebook.

**Take-home for Day 2:** every VS Code companion above shared one local session —
Day 2 opens a fresh one, since none of its techniques need Delta at all.